# v005 · Trích feature (AV-HuBERT Large, crop miệng, khung LIPINC-V2, XLS-R 5 nhóm lớp, waveform)

Mỗi part (`PART` = 1, 2, 3) chạy **ba phiên** bản gốc (+ 5 phiên biến thể v005), mỗi phiên dưới 12 giờ và Output dưới 20 GB. Đặt tên notebook Kaggle đúng `SESSION_NAME` (`vn-av-df-prepare-p<PART>-<SESSION>`) để notebook train tìm được Output:
1. `SESSION = "landmarks"` (Accelerator: None, không tốn quota GPU): dlib dò mặt (part 1 v003: 8,7 giờ).
2. Sau phiên 1 của **cùng part**, chạy song song hai phiên, cả hai attach Output phiên 1 vào `OTHER_CACHES`:
   - `SESSION = "cpu"` (None): `mouth`, `lipinc`, `wave`.
   - `SESSION = "gpu"` (T4 ×2): `avhubert` (ghi thêm `avhubert/sync_profile.json` cho nhãn mismatch v004), `xlsr`.
3. v005, mỗi biến thể dữ liệu (`settings.VARIANTS`) một phiên T4 ×2, attach Output phiên `landmarks` của cùng part vào `OTHER_CACHES`; tạo video biến thể (giữ số frame) rồi trích `avhubert`, `mouth`, `xlsr` trên bản đó:
   - `SESSION = "c480"`, `"c360"`: nén lại mọi mẫu (480p libx264 CRF 30 + AAC 64k; 360p CRF 35 + AAC 32k), dùng khi train + test.
   - `SESSION = "pr"`: real / sham / voice qua xử lý lành tính giống đường ống generator (x264 CRF 8 → codec / co giãn → x264 CRF 18, tiếng giữ nguyên, nhãn giữ nguyên); train / validation 4 họ xử lý, test một nửa họ đã thấy + một nửa 3 họ chưa thấy (`clips/reencode.json` → `chains`). Âm khó khi train + phép thử "AI thật hay vết đường ống".
   - `SESSION = "w360"` (VP9 360p + Opus), `"c288"` (288p): chỉ mẫu test, mức không có khi train (ngắn).
   Xong thì xoá clip khỏi Output (giữ `clips/reencode.json`); dở giữa chừng thì giữ để phiên sau (`PREVIOUS_CACHE`) không làm lại.

Hết giờ giữa chừng: chạy lại cùng `SESSION`, attach Output phiên dở vào `PREVIOUS_CACHE`. Cell "Log setup" ghi cách cài phiên vào `cache/vn-av-df-data/setup/`; run W&B `prepare` của phiên đính kèm luôn (config `setup`, bảng `setup/*`, artifact `setup-log`).

In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json, shutil
SESSION_START = time.time()
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"  # Sửa thành repository của bạn.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
WORKERS = Path("/kaggle/temp/vn-av-df-workers")  # Môi trường worker: ngoài Output 20 GB.
# Log cài đặt của phiên (git, pip, worker): ghi đủ ra file, cell "Log setup" đưa lên W&B + Output.
LOGS = Path("/kaggle/temp/vn-av-df-logs")
LOGS.mkdir(parents=True, exist_ok=True)
TIMINGS = {}
def logged(name, command, **kw):
    tick = time.time()
    path = LOGS / f"{name}.log"
    with open(path, "a", encoding="utf-8") as log:
        log.write("$ " + " ".join(map(str, command)) + "\n")
        log.flush()
        code = subprocess.run(list(map(str, command)), stdout=log, stderr=subprocess.STDOUT, **kw).returncode
    TIMINGS[name] = round(TIMINGS.get(name, 0) + time.time() - tick, 1)
    if code:
        print("\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
        raise RuntimeError(f"{name} lỗi (mã {code}); log đầy đủ: {path}")
if not ROOT.exists():
    logged("git-clone", ["git", "clone", REPO_URL, str(ROOT)])
# Weight và repo upstream (vài GB) ở /kaggle/temp: Output (giới hạn 20 GB) không bao giờ chứa
# chúng, kể cả khi phiên lỗi giữa chừng (bước dọn Output cuối không chạy).
for name in ("weights", "external"):
    (Path("/kaggle/temp/vn-av-df-heavy") / name).mkdir(parents=True, exist_ok=True)
    if not (ROOT / name).exists():
        (ROOT / name).symlink_to(Path("/kaggle/temp/vn-av-df-heavy") / name, target_is_directory=True)
sys.path.insert(0, str(ROOT))
logged("pip-install", [sys.executable, "-m", "pip", "install", "-e", str(ROOT) + "[data,training]"])
import settings
settings.bootstrap()

In [ ]:
# W&B: log quá trình (setup từng phiên, trích feature, nhãn, train, test, báo cáo) làm bằng chứng.
# Project PRIVATE. Thêm secret WANDB_API_KEY: Add-ons → Secrets. Không đẩy ảnh / video người thật.
USE_WANDB = True
WANDB_PROJECT = "vn-av-df-v005"  # v004: "vn-av-df-v004", v003: "vn-av-df-v003" (giữ nguyên, không ghi chồng)
WANDB_ENTITY = "vn-av-df-capstone"  # team trong org vn-av-df-capstone-org; key tạo trong org đó
if USE_WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_DIR"] = "/kaggle/temp"  # file tạm W&B ở ngoài Output
    os.makedirs(os.environ["WANDB_DIR"], exist_ok=True)
WANDB = {"project": WANDB_PROJECT, "entity": WANDB_ENTITY} if USE_WANDB else None

In [ ]:
PART = 1               # 1, 2, 3
SESSION = "landmarks"  # "landmarks" (CPU) → rồi song song "cpu" (CPU), "gpu", "c480", "c360", "pr", "w360", "c288" (T4 ×2)
# Slug dataset Kaggle của từng part (bản có sync_syncnet.json); sửa nếu upload tên khác.
DATASETS = {1: "linhxm/vn-av-df-data-part1-trim04-fake",
            2: "linhxm/vn-av-df-data-part2-fake",
            3: "linhxm/vn-av-df-data-part3-fake"}
DATASET_PARTS = [Path(f"/kaggle/input/datasets/{DATASETS[PART]}")]
# Hết giờ giữa chừng: chạy lại cùng SESSION, attach Output phiên dở vào đây (mẫu đã có được bỏ qua).
PREVIOUS_CACHE = None  # Path("/kaggle/input/<Output phiên dở>/vn-av-df-forensics/cache/vn-av-df-data")
# Phiên "cpu" / "gpu" / biến thể: Output phiên "landmarks" CỦA CÙNG PART (đường dẫn ở panel Input).
OTHER_CACHES = []      # vd. [Path("/kaggle/input/notebooks/linhxm/vn-av-df-prepare-p1-landmarks/vn-av-df-forensics/cache/vn-av-df-data")]
SESSION_NAME = f"prepare-p{PART}-{SESSION}"
MODELS = settings.MODELS + settings.ABLATIONS
import torch
from vn_av_df.dataset import part_selection
from vn_av_df.training.prepare import encoders_for
GPU = torch.cuda.is_available()
settings.DEVICE = "cuda" if GPU else "cpu"
cfg = settings.config()
CHOOSE = {
    "landmarks": lambda e: e == "landmarks",
    "cpu": lambda e: e != "landmarks" and cfg["encoders"][e].get("cpu_only"),
    "gpu": lambda e: not cfg["encoders"][e].get("cpu_only"),
    # Biến thể (settings.VARIANTS: nén, PR, mức chỉ test): landmark dùng lại của bản gốc; model v005 chỉ cần 3 encoder.
    **{v: (lambda e: e in ("avhubert", "mouth", "xlsr")) for v in settings.VARIANTS},
}
ENCODERS = [e for e in encoders_for(MODELS) if CHOOSE[SESSION](e)]  # hoặc chọn tay
if SESSION in settings.VARIANTS:
    cfg["variant"] = dict(settings.VARIANTS[SESSION])
if (SESSION == "gpu" or SESSION in settings.VARIANTS) and not GPU:
    raise RuntimeError(f'SESSION = "{SESSION}" cần Accelerator GPU (T4 ×2)')
if SESSION in ("landmarks", "cpu") and GPU:
    print(f"CẢNH BÁO: phiên {SESSION} chỉ cần CPU, chọn Accelerator None để không tốn quota GPU")
if SESSION != "landmarks" and not OTHER_CACHES:
    print("CẢNH BÁO: OTHER_CACHES rỗng, worker sẽ tự dò landmark bằng dlib (chậm)")
print("Phiên", SESSION_NAME, "→ ENCODERS =", ENCODERS)
root, parts = part_selection(DATASET_PARTS)
cfg["dataset"], cfg["dataset_parts"] = str(root), parts
cfg["prepare_devices"] = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or [None]
cfg["wandb"] = WANDB
cfg["session"] = SESSION_NAME
OUTPUT_CACHE = Path(cfg["cache"])  # ghi vào đây; đọc thêm từ OTHER_CACHES
cfg["run_group"] = OUTPUT_CACHE.name  # cùng group W&B với run prepare
if PREVIOUS_CACHE and not OUTPUT_CACHE.exists():
    shutil.copytree(PREVIOUS_CACHE, OUTPUT_CACHE)
cfg["cache"] = [str(OUTPUT_CACHE), *map(str, OTHER_CACHES)]

In [ ]:
from vn_av_df.cli import setup
tick = time.time()
ASSETS = setup(cfg, generators=[], encoders=ENCODERS)  # chỉ tải thứ ENCODERS cần
TIMINGS["assets"] = round(time.time() - tick, 1)
print(ASSETS)
if {"landmarks", "avhubert"} & set(ENCODERS):
    # Cùng env cho cả hai stage; phiên CPU cài không cần GPU (stage landmarks chỉ dùng dlib).
    logged("worker-avhubert", [sys.executable, str(ROOT / "environments/bootstrap_worker.py"), "avhubert",
                               "--root", str(ROOT), "--env-root", str(WORKERS)] + ([] if GPU else ["--allow-cpu"]))
    cfg["encoders"]["avhubert"]["python"] = str(WORKERS / ".venv-avhubert/bin/python")

## Log setup

In [ ]:
SETUP_FOLDER = OUTPUT_CACHE / "setup"
SETUP_INFO = {"part": PART, "session": SESSION, "encoders": ENCODERS, "datasets": [str(p) for p in DATASET_PARTS],
              "other_caches": [str(p) for p in OTHER_CACHES], "previous_cache": str(PREVIOUS_CACHE or ""),
              "assets": ASSETS}

# Ghi cách cài phiên vào Output: commit, phần cứng, gói pip của kernel và từng worker, log cài (pip,
# worker), dữ liệu / cache / tài sản gắn vào. Run W&B của phiên (prepare / train / labels / report) tự
# đính kèm: config `setup`, bảng `setup/*`, artifact `setup-log`. Mở run nào cũng biết nó được cài thế nào.
from vn_av_df.setup_log import log_setup
cfg["setup"] = log_setup(cfg, SESSION_NAME, SETUP_FOLDER, logs=sorted(LOGS.glob("*.log")), env_root=WORKERS,
                         info=SETUP_INFO, timings=TIMINGS)

In [ ]:
from vn_av_df.training.prepare import prepare
from vn_av_data.common.runtime import prune_output
try:
    print(prepare(cfg, ENCODERS))
    if cfg.get("variant"):  # xong hết: bỏ clip nén khỏi Output (train chỉ đọc đặc trưng)
        clips = sorted((OUTPUT_CACHE / "clips").glob("*.mp4"))
        for clip in clips:
            clip.unlink()
        print("Đã xoá", len(clips), "clip nén; giữ clips/reencode.json")
finally:  # Output chỉ giữ cache (kèm log setup), kể cả khi lỗi giữa chừng (phiên sau dùng PREVIOUS_CACHE)
    print("Đã xoá khỏi Output:", len(prune_output(ROOT, [str(OUTPUT_CACHE.relative_to(ROOT))])), "mục")